# RAMM — Unified Backtest Notebook
### LSTM vs LightGBM Tree | Multi-Asset | Full Transfer Testing

**18 experiments, fully automated:**

| Phase | Runs | Description |
|-------|------|-------------|
| Phase 1 | 6 | Native: BTC/ETH/SOL × LSTM/Tree on own asset |
| Phase 2 | 12 | Transfer: 6 pairs × 2 models (frozen weights) |

**Cells:** 0 config → 1–9 definitions → 10 load models → 11 Phase 1 → 12 save → 13 compare → 14 microstructure → 15 transfer → 16 figures


In [1]:
%pip install pandas scipy lightgbm matplotlib cupy-cuda12x


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch, numpy as np, pandas as pd, glob, os, gc, time, json, subprocess, itertools
import warnings; warnings.filterwarnings("ignore")
from scipy import stats as scipy_stats

# == PER-ASSET MODEL PATHS ====================================================
# Each asset has its OWN trained LSTM and Tree model.
LSTM_MODEL_PATHS = {
    'BTC': '/workspace/data/14-8-26(btc)/lstm/best_model (1).pth',
    'ETH': '/workspace/data/15-8-26(eth)/lstm/best_model (1).pth',
    'SOL': '/workspace/data/15-8-26(sol)/lstm/best_model (1).pth',
}

TREE_MODEL_PATHS = {
    'BTC': {
        'vol':      '/workspace/data/14-8-26(btc)/tree/vol_model_final.txt',
        'ofi':      '/workspace/data/14-8-26(btc)/tree/ofi_model_final.txt',
        'features': '/workspace/data/14-8-26(btc)/tree/selected_features.json',
    },
    'ETH': {
        'vol':      '/workspace/data/15-8-26(eth)/tree/vol_model_final.txt',
        'ofi':      '/workspace/data/15-8-26(eth)/tree/ofi_model_final.txt',
        'features': '/workspace/data/15-8-26(eth)/tree/selected_features.json',
    },
    'SOL': {
        'vol':      '/workspace/data/15-8-26(sol)/tree/vol_model_final.txt',
        'ofi':      '/workspace/data/15-8-26(sol)/tree/ofi_model_final.txt',
        'features': '/workspace/data/15-8-26(sol)/tree/selected_features.json',
    },
}

# Per-asset data folders — each asset has its own folder of .pt files
# Adjust these paths to where your data is on Colab/local machine
DATA_PATHS = {
    'BTC': '/workspace/data/BTC_Processed',   # folder containing tensor_OFI_Enhanced_2024-*.pt for BTC
    'ETH': '/workspace/data/ETH_Processed',   # folder containing tensor_OFI_Enhanced_2024-*.pt for ETH
    'SOL': '/workspace/data/SOL_Processed',   # folder containing tensor_OFI_Enhanced_2024-*.pt for SOL
}
# If ALL assets are in ONE flat folder with asset name in filename:
#   DATA_PATHS = {'BTC': '/workspace/data', 'ETH': '/workspace/data', 'SOL': '/workspace/data'}
#   and set ASSET_GLOB = {'BTC': 'BTC', 'ETH': 'ETH', 'SOL': 'SOL'}

YEAR_FILTER = '2024'   # filter filenames containing this string; None = all files

# Pattern that must appear in filename (within the per-asset folder)
# Since files are named tensor_OFI_Enhanced_2024-xx.pt with no asset name,
# use 'tensor_OFI_Enhanced' to match all files in each folder.
ASSET_GLOB = {
    'BTC': 'tensor_OFI_Enhanced',
    'ETH': 'tensor_OFI_Enhanced',
    'SOL': 'tensor_OFI_Enhanced',
}
ASSETS = ['BTC', 'ETH', 'SOL']

INITIAL_CASH   = 10_000_000
MAKER_FEE      = 0.000
MAX_INVENTORY  = 5.0
LATENCY_MS     = 10
ROWS_PER_CHUNK = 50_000_000

ASSET_CONFIG = {
    'BTC': {'gamma_base': 0.001,  'vol_baseline': 1.0, 'tick_size': 0.1000},
    'ETH': {'gamma_base': 0.001,  'vol_baseline': 0.8, 'tick_size': 0.0905},
    'SOL': {'gamma_base': 0.001, 'vol_baseline': 1.2, 'tick_size': 0.0572},
}


# == NORMALIZATION PARAMS =====================================================
# Backtesting (out-of-sample) params — same as normalization_params.json
NORM_PARAMS_TEST = {
    "BTC": {"log_price_mean": 10.861594, "log_price_std": 0.010742},
    "ETH": {"log_price_mean":  7.954236, "log_price_std": 0.011872},
    "SOL": {"log_price_mean":  4.771956, "log_price_std": 0.018789},
}
# Training (in-sample) params — used to quantify distribution shift
NORM_PARAMS_TRAIN = {
    "BTC": {"log_price_mean": 10.973000, "log_price_std": 0.012386},
    "ETH": {"log_price_mean":  7.519578, "log_price_std": 0.007662},
    "SOL": {"log_price_mean":  3.355671, "log_price_std": 0.017100},
}
print("Configuration loaded")
print(f"ASSETS: {ASSETS} | DATA_PATHS configured | YEAR_FILTER: {YEAR_FILTER}")
for a in ASSETS:
    print(f"  {a} LSTM: {LSTM_MODEL_PATHS[a]}")
    print(f"  {a} Tree: {TREE_MODEL_PATHS[a]['vol']}")


Configuration loaded
ASSETS: ['BTC', 'ETH', 'SOL'] | DATA_PATHS configured | YEAR_FILTER: 2024
  BTC LSTM: /workspace/data/14-8-26(btc)/lstm/best_model (1).pth
  BTC Tree: /workspace/data/14-8-26(btc)/tree/vol_model_final.txt
  ETH LSTM: /workspace/data/15-8-26(eth)/lstm/best_model (1).pth
  ETH Tree: /workspace/data/15-8-26(eth)/tree/vol_model_final.txt
  SOL LSTM: /workspace/data/15-8-26(sol)/lstm/best_model (1).pth
  SOL Tree: /workspace/data/15-8-26(sol)/tree/vol_model_final.txt


## Cell 1 — GammaMapper


In [3]:
import numpy as np

class GammaMapper:
    """Maps model predictions (vol, OFI) to dynamic risk aversion γ_t."""

    def __init__(self, gamma_base=0.1, vol_baseline=1.0, ofi_sensitivity=5.0,
                 vol_sensitivity=1.0, gamma_min=0.01, gamma_max=1.0):
        self.gamma_base      = gamma_base
        self.vol_baseline    = vol_baseline
        self.ofi_sensitivity = ofi_sensitivity
        self.vol_sensitivity = vol_sensitivity
        self.gamma_min       = gamma_min
        self.gamma_max       = gamma_max

    def compute_gamma(self, volatility_pred, ofi_pred):
        vol_factor = max(0.5, 1.0 + self.vol_sensitivity * (volatility_pred / self.vol_baseline - 1.0))
        ofi_factor = 1.0 + self.ofi_sensitivity * abs(ofi_pred)
        gamma_t    = self.gamma_base * vol_factor * ofi_factor
        return float(np.clip(gamma_t, self.gamma_min, self.gamma_max))

    def get_regime_label(self, gamma_t):
        if gamma_t < 0.05:  return "AGGRESSIVE"
        if gamma_t < 0.15:  return "NORMAL"
        if gamma_t < 0.30:  return "CAUTIOUS"
        return "DEFENSIVE"

print("✅ GammaMapper defined")


✅ GammaMapper defined


## Cell 2 — Price Denormalization


In [4]:
import numpy as np, json as _json, os

# Real backtesting (out-of-sample) normalization params
# BTC/ETH/SOL log-price stats computed from the test-period data
NORM_PARAMS_FALLBACK = {
    "BTC": {"log_price_mean": 10.861594, "log_price_std": 0.010742},
    "ETH": {"log_price_mean":  7.954236, "log_price_std": 0.011872},
    "SOL": {"log_price_mean":  4.771956, "log_price_std": 0.018789},
}

def load_normalization_params(params_path="normalization_params.json"):
    if os.path.exists(params_path):
        with open(params_path) as f:
            return _json.load(f)
    return None

def get_norm_params(asset, params_path="normalization_params.json"):
    saved = load_normalization_params(params_path)
    if saved and asset.upper() in saved:
        return saved[asset.upper()]
    fallback = NORM_PARAMS_FALLBACK.get(asset.upper(), NORM_PARAMS_FALLBACK["BTC"])
    print(f"   Using fallback norm params for {asset}: {fallback}")
    return fallback

def denormalize_price(normalized_price, log_price_mean, log_price_std):
    return np.exp(normalized_price * log_price_std + log_price_mean)

def make_denormalize_fn(asset, params_path="normalization_params.json"):
    p = get_norm_params(asset, params_path)
    mean, std = p["log_price_mean"], p["log_price_std"]
    def _denorm(x): return np.exp(x * std + mean)
    print(f"   Denorm fn for {asset}: exp(x * {std} + {mean})")
    return _denorm

print("✅ Denormalization utilities loaded")


✅ Denormalization utilities loaded


## Cell 3 — Avellaneda-Stoikov Market Maker


In [5]:
import numpy as np

class AvellanedaStoikovMarketMaker:
    """Complete AS framework with RAMM-style dynamic risk aversion."""

    def __init__(self, gamma_mapper, k=1.5, T=3600, tick_size=0.01, max_inventory=5.0):
        self.gamma_mapper = gamma_mapper
        self.k            = k
        self.T            = T
        self.tick_size    = tick_size
        self.max_inventory = max_inventory

    def compute_reservation_price(self, mid_price, inventory, gamma_t, sigma, tau):
        return mid_price - inventory * gamma_t * (sigma ** 2) * tau

    def compute_optimal_spread(self, gamma_t):
        return (1.0 / gamma_t) * np.log(1.0 + gamma_t / self.k)

    def compute_quotes(self, mid_price, inventory, volatility_pred, ofi_pred, time_elapsed):
        tau         = max(1.0, self.T - time_elapsed)
        gamma_t     = self.gamma_mapper.compute_gamma(volatility_pred, ofi_pred)
        res         = self.compute_reservation_price(mid_price, inventory, gamma_t, volatility_pred, tau)
        spread_half = self.compute_optimal_spread(gamma_t)
        bid         = self._round_to_tick(res - spread_half)
        ask         = self._round_to_tick(res + spread_half)
        min_spread  = 2 * self.tick_size
        if ask - bid < min_spread:
            bid = mid_price - min_spread / 2
            ask = mid_price + min_spread / 2
        return {"bid": bid, "ask": ask, "mid": mid_price, "gamma": gamma_t,
                "reservation_price": res, "spread": ask - bid,
                "regime": self.gamma_mapper.get_regime_label(gamma_t),
                "volatility": volatility_pred, "ofi": ofi_pred}

    def _round_to_tick(self, price):
        return np.round(price / self.tick_size) * self.tick_size

print("✅ AvellanedaStoikovMarketMaker defined")


✅ AvellanedaStoikovMarketMaker defined


## Cell 5 — TreeMarketModel + Feature Engineering


In [6]:
import lightgbm as lgb
import numpy as np, pandas as pd, json

COLUMN_NAMES    = ["price","volume","spread","OFI","direction","volatility","price_change"]
LAG_RANGE       = list(range(1, 31))
ROLLING_WINDOWS = [5, 10, 15, 20, 30, 50]
EWMA_SPANS      = [5, 10, 15, 20, 30, 50]
WARMUP_ROWS     = 50

def build_features(data: np.ndarray, selected_features=None) -> pd.DataFrame:
    """Build ~103-column feature set. If selected_features given, compute only those."""
    df   = pd.DataFrame(data, columns=COLUMN_NAMES).astype(np.float32)
    need = (lambda c: True) if selected_features is None else (lambda c: c in set(selected_features))
    for lag in LAG_RANGE:
        if need(f"OFI_lag_{lag}"):   df[f"OFI_lag_{lag}"]   = df["OFI"].shift(lag).astype(np.float32)
        if need(f"vol_lag_{lag}"):   df[f"vol_lag_{lag}"]   = df["volatility"].shift(lag).astype(np.float32)
    for w in ROLLING_WINDOWS:
        if need(f"OFI_rolling_mean_{w}"): df[f"OFI_rolling_mean_{w}"] = df["OFI"].rolling(w,min_periods=w).mean().astype(np.float32)
        if need(f"OFI_rolling_std_{w}"):  df[f"OFI_rolling_std_{w}"]  = df["OFI"].rolling(w,min_periods=w).std().astype(np.float32)
        if need(f"vol_rolling_mean_{w}"): df[f"vol_rolling_mean_{w}"] = df["volatility"].rolling(w,min_periods=w).mean().astype(np.float32)
        if need(f"vol_rolling_std_{w}"):  df[f"vol_rolling_std_{w}"]  = df["volatility"].rolling(w,min_periods=w).std().astype(np.float32)
    for s in EWMA_SPANS:
        if need(f"OFI_ewma_{s}"): df[f"OFI_ewma_{s}"] = df["OFI"].ewm(span=s,adjust=False).mean().astype(np.float32)
        if need(f"vol_ewma_{s}"): df[f"vol_ewma_{s}"] = df["volatility"].ewm(span=s,adjust=False).mean().astype(np.float32)
    df = df.iloc[WARMUP_ROWS:].reset_index(drop=True)
    if selected_features:
        keep = list(selected_features) + [c for c in ["volatility","OFI"] if c not in selected_features]
        df   = df[[c for c in keep if c in df.columns]]
    return df.astype(np.float32)


class TreeMarketModel:
    """
    Drop-in replacement for LSTMMarketModel.
    predict_from_window(window: np.ndarray[W,7]) -> (vol_pred, ofi_pred)
    """

    def __init__(self, vol_model, ofi_model, selected_features):
        self.vol_model         = vol_model
        self.ofi_model         = ofi_model
        self.selected_features = selected_features
        self._predict_vol      = vol_model.predict
        self._predict_ofi      = ofi_model.predict

    @classmethod
    def from_saved(cls, vol_path, ofi_path, features_path):
        vol_b = lgb.Booster(model_file=vol_path)
        ofi_b = lgb.Booster(model_file=ofi_path)
        with open(features_path) as f:
            sel = json.load(f)["selected_features"]
        print(f"✅ TreeMarketModel loaded  — {len(sel)} features")
        print(f"   Vol: {vol_path}")
        print(f"   OFI: {ofi_path}")
        return cls(vol_b, ofi_b, sel)

    def predict_from_window(self, recent_rows: np.ndarray):
        df    = build_features(recent_rows, selected_features=self.selected_features)
        if df.empty: return 0.0, 0.0
        fcols = [c for c in self.selected_features if c not in ("volatility","OFI") and c in df.columns]
        row   = df[fcols].iloc[[-1]].values.astype(np.float32)
        return float(self._predict_vol(row)[0]), float(self._predict_ofi(row)[0])

    def __repr__(self):
        return f"TreeMarketModel(features={len(self.selected_features)})"

print("✅ TreeMarketModel + build_features defined")


✅ TreeMarketModel + build_features defined


## Cell 6 — Shared Vectorized Engine (ChunkedBacktestEngine)


In [7]:
import numpy as cp; CUPY_AVAILABLE = False  # FORCED NUMPY CPU FALLBACK
print('CuPy explicitly disabled -- CPU vectorization forced')
import subprocess, numpy as np, pandas as pd, os

def get_gpu_memory():
    try:
        r = subprocess.run(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],
                           capture_output=True, text=True)
        return int(r.stdout.strip())
    except: return 0


class ChunkedBacktestEngine:
    # FIX 6: tracks both position_count (for AS formula) and inventory_units (for MTM)
    def __init__(self, mm_strategy, maker_fee=0.0, initial_cash=100_000,
                 max_inventory=5.0, fill_notional_usd=10_000):
        self.mm                = mm_strategy
        self.maker_fee         = maker_fee
        self.initial_cash      = initial_cash
        self.max_inventory     = max_inventory       # position count limit (e.g. 2)
        self.fill_notional_usd = fill_notional_usd   # dollar value per fill
        self.inventory         = 0.0   # position count  -- used in AS reservation price
        self.inventory_units   = 0.0   # actual units    -- used in MTM & liquidation
        self.cash              = initial_cash
        self.metrics = {'total_fills': 0, 'adverse_selections': 0, 'total_fees_paid': 0,
                        'max_inventory': 0, 'bid_fills': 0, 'ask_fills': 0}

    def get_summary_metrics(self):
        fills = self.metrics['total_fills']
        return {
            'total_fills':          fills,
            'adverse_selections':   self.metrics['adverse_selections'],
            'adverse_selection_pct': (self.metrics['adverse_selections']/fills*100 if fills > 0 else 0),
            'total_fees_paid':      self.metrics['total_fees_paid'],
            'max_inventory':        self.metrics['max_inventory'],
            'bid_fills':            self.metrics['bid_fills'],
            'ask_fills':            self.metrics['ask_fills'],
            'final_inventory':      self.inventory,        # position count
            'final_inventory_units': self.inventory_units, # actual units (BTC/ETH/SOL)
            'final_cash':           self.cash,
        }


def vectorized_backtest_chunk(chunk_data, vol_preds, ofi_preds, engine,
                               gamma_mapper, mm_strategy, global_idx_start,
                               total_rows, mini_batch=1000, denorm_params=None,
                               latency_rows=1):
    # FIX 5: latency_rows = int(inference_time * rows_per_second)
    # FIX 6: fill_qty = fill_notional_usd / real_price (dollar-neutral)
    # FIX 7: spread_captured per fill for PnL decomposition
    N = len(chunk_data)
    if CUPY_AVAILABLE:
        mid_prices = cp.asarray(chunk_data[:,0], dtype=cp.float64)
        vol        = cp.asarray(vol_preds,        dtype=cp.float64)
        ofi        = cp.asarray(ofi_preds,        dtype=cp.float64)
    else:
        mid_prices = cp.array(chunk_data[:,0], dtype=cp.float64)
        vol        = cp.array(vol_preds,        dtype=cp.float64)
        ofi        = cp.array(ofi_preds,        dtype=cp.float64)

    if denorm_params is not None:
        _lp_mean = denorm_params['log_price_mean']
        _lp_std  = denorm_params['log_price_std']
        real_mid_prices = cp.exp(mid_prices * _lp_std + _lp_mean)
    else:
        real_mid_prices = mid_prices

    vol_factor = cp.maximum(0.5, 1.0 + gamma_mapper.vol_sensitivity * (vol / gamma_mapper.vol_baseline - 1.0))
    ofi_factor = 1.0 + gamma_mapper.ofi_sensitivity * cp.abs(ofi)
    gammas     = cp.clip(gamma_mapper.gamma_base * vol_factor * ofi_factor,
                         gamma_mapper.gamma_min, gamma_mapper.gamma_max)

    spreads_half   = (1.0 / gammas) * cp.log(1.0 + gammas / mm_strategy.k)
    global_indices = global_idx_start + cp.arange(N, dtype=cp.float64)
    time_elapsed   = (global_indices / total_rows) * mm_strategy.T
    tau            = cp.maximum(1.0, mm_strategy.T - time_elapsed)
    rand_bid       = cp.random.rand(N)
    rand_ask       = cp.random.rand(N)
    A = 1.0; k = mm_strategy.k; dt = 0.0001
    ts = mm_strategy.tick_size; min_spread = 5 * ts

    # FIX 5: realistic latency -- roll by actual inferred staleness rows
    # Old code: cp.roll(vol, 1)  -- only 1 row = ~55ms for BTC
    # New code: cp.roll(vol, latency_rows) where latency_rows ~= 457 for BTC
    lr = max(1, min(int(latency_rows), N - 1))
    vol = cp.roll(vol, lr)
    ofi = cp.roll(ofi, lr)

    # FIX 6: separate position count (for AS formula) from inventory_units (for MTM)
    inventory        = engine.inventory        # position count (-MAX..+MAX)
    inventory_units  = engine.inventory_units  # actual asset units
    cash             = engine.cash

    pnl_chunks = []; pos_chunks = []; fill_list = []
    gamma_history = []

    for mb_s in range(0, N, mini_batch):
        mb_e    = min(mb_s + mini_batch, N)
        mb_size = mb_e - mb_s
        mb_mid  = mid_prices[mb_s:mb_e]; mb_gamma = gammas[mb_s:mb_e]
        mb_spread = spreads_half[mb_s:mb_e]; mb_tau = tau[mb_s:mb_e]
        mb_vol  = vol[mb_s:mb_e]; mb_rb = rand_bid[mb_s:mb_e]; mb_ra = rand_ask[mb_s:mb_e]
        mb_real_mid = real_mid_prices[mb_s:mb_e]

        # AS reservation price uses position count (consistent scale across assets)
        res  = mb_mid - inventory * mb_gamma * (mb_vol**2) * mb_tau
        bids = cp.round((res - mb_spread) / ts) * ts
        asks = cp.round((res + mb_spread) / ts) * ts
        tight = (asks - bids) < min_spread
        bids  = cp.where(tight, mb_mid - min_spread/2, bids)
        asks  = cp.where(tight, mb_mid + min_spread/2, asks)

        if denorm_params is not None:
            real_bids = cp.exp(bids * _lp_std + _lp_mean)
            real_asks = cp.exp(asks * _lp_std + _lp_mean)
        else:
            real_bids = bids; real_asks = asks

        future_mid = cp.roll(mb_mid, -1)
        future_mid[-1] = mb_mid[-1]

        prob_bid = 1.0 - cp.exp(-A * cp.exp(-k * cp.abs(mb_mid - bids)) * dt)
        prob_ask = 1.0 - cp.exp(-A * cp.exp(-k * cp.abs(asks - mb_mid)) * dt)

        bid_fills = (future_mid <= bids) | (mb_rb < prob_bid)
        ask_fills = ((future_mid >= asks) | (mb_ra < prob_ask)) & ~bid_fills

        if CUPY_AVAILABLE:
            bid_fills_cpu   = cp.asnumpy(bid_fills);  ask_fills_cpu = cp.asnumpy(ask_fills)
            bids_cpu        = cp.asnumpy(bids);       asks_cpu      = cp.asnumpy(asks)
            mb_mid_cpu      = cp.asnumpy(mb_mid);     mb_gamma_cpu  = cp.asnumpy(mb_gamma)
            real_bids_cpu   = cp.asnumpy(real_bids);  real_asks_cpu = cp.asnumpy(real_asks)
            mb_real_mid_cpu = cp.asnumpy(mb_real_mid)
        else:
            bid_fills_cpu   = bid_fills;  ask_fills_cpu = ask_fills
            bids_cpu        = bids;       asks_cpu      = asks
            mb_mid_cpu      = mb_mid;     mb_gamma_cpu  = mb_gamma
            real_bids_cpu   = real_bids;  real_asks_cpu = real_asks
            mb_real_mid_cpu = mb_real_mid

        fill_idx        = np.where(bid_fills_cpu | ask_fills_cpu)[0]
        # pos_timeline  = position count snapshot (for reporting)
        # inv_timeline  = inventory_units snapshot (for MTM)
        pos_timeline  = np.full(mb_size, inventory,       dtype=np.float64)
        inv_timeline  = np.full(mb_size, inventory_units, dtype=np.float64)
        cash_timeline = np.full(mb_size, cash,            dtype=np.float64)

        for fi in fill_idx:
            side       = 'bid' if bid_fills_cpu[fi] else 'ask'
            norm_price = float(bids_cpu[fi] if side=='bid' else asks_cpu[fi])
            real_price = float(real_bids_cpu[fi] if side=='bid' else real_asks_cpu[fi])
            real_mid_now = float(mb_real_mid_cpu[fi])

            # FIX 6: dollar-neutral fill quantity
            fill_qty = engine.fill_notional_usd / real_price  # e.g. 10000/52135 = 0.192 BTC

            # Inventory limit uses position count (unchanged scale across assets)
            can_trade = ((side=='bid' and inventory < engine.max_inventory) or
                         (side=='ask' and inventory > -engine.max_inventory))
            if not can_trade: continue

            fee = fill_qty * real_price * engine.maker_fee  # = 0 when MAKER_FEE = 0
            future_norm_mid = float(mb_mid_cpu[min(fi + 10, mb_size - 1)])
            markout    = (future_norm_mid - norm_price) * (1 if side == 'bid' else -1)
            is_adverse = markout < -0.01

            # FIX 7: spread captured = distance of fill price from mid, in dollars
            spread_captured = abs(real_price - real_mid_now) * fill_qty

            if side == 'bid':
                inventory       += 1.0         # position count +1
                inventory_units += fill_qty     # actual units   +fill_qty
                cash            -= (fill_qty * real_price - fee)  # ~= -FILL_NOTIONAL
                engine.metrics['bid_fills'] += 1
            else:
                inventory       -= 1.0
                inventory_units -= fill_qty
                cash            += (fill_qty * real_price + fee)   # ~= +FILL_NOTIONAL
                engine.metrics['ask_fills'] += 1

            pos_timeline[fi:]  = inventory
            inv_timeline[fi:]  = inventory_units
            cash_timeline[fi:] = cash

            fill_list.append({
                't': global_idx_start + mb_s + int(fi),
                'side': side,
                'price': real_price,
                'fee': fee,
                'quantity': fill_qty,              # actual units (dollar-neutral)
                'inventory': inventory,            # position count
                'inventory_units': inventory_units,
                'spread_captured': spread_captured, # FIX 7
                'markout': markout,
                'adverse': bool(is_adverse),
            })
            engine.metrics['total_fills']     += 1
            engine.metrics['total_fees_paid'] += fee
            if is_adverse: engine.metrics['adverse_selections'] += 1
            if abs(inventory) > engine.metrics['max_inventory']:
                engine.metrics['max_inventory'] = abs(inventory)

        # MTM uses inventory_units * real_mid_price (dollar-correct)
        inv_gpu  = cp.asarray(inv_timeline)
        cash_gpu = cp.asarray(cash_timeline)
        mtm = cash_gpu + inv_gpu * mb_real_mid
        pnl_chunks.append((cp.asnumpy(mtm) if CUPY_AVAILABLE else mtm) - engine.initial_cash)
        pos_chunks.append(pos_timeline)   # position count for peak/avg reporting
        gamma_history.append(mb_gamma_cpu)

    engine.inventory       = inventory
    engine.inventory_units = inventory_units
    engine.cash            = cash

    pnl_all   = np.concatenate(pnl_chunks)
    pos_all   = np.concatenate(pos_chunks)   # position count (0..MAX_INVENTORY)
    gamma_all = np.concatenate(gamma_history)
    return (pnl_all, pos_all, gamma_all), fill_list

print('ChunkedBacktestEngine + vectorized_backtest_chunk -- FIX5 latency, FIX6 dollar-neutral, FIX7 spread_captured')

CuPy explicitly disabled -- CPU vectorization forced
ChunkedBacktestEngine + vectorized_backtest_chunk -- FIX5 latency, FIX6 dollar-neutral, FIX7 spread_captured


## Cell 8 — TreeBacktestRunner
> Uses full-chunk vectorized LightGBM inference (~100x faster than row-by-row).


In [8]:
import time, numpy as np

class TreeBacktestRunner:
    """
    Tree-specific backtest runner.
    Uses vectorized LightGBM inference (full chunk → single predict() call)
    + same vectorized CuPy engine as LSTM.
    ~50-100x faster than row-by-row tree inference.
    """

    def __init__(self, model, initial_cash=100_000, maker_fee=0.0002, max_inventory=5.0):
        self.model         = model
        self.initial_cash  = initial_cash
        self.maker_fee     = maker_fee
        self.max_inventory = max_inventory

    def _batch_tree_inference(self, data_chunk):
        """Build features for entire chunk at once → single LightGBM predict call."""
        df = build_features(data_chunk, selected_features=self.model.selected_features)
        if df.empty:
            return (np.zeros(len(data_chunk), dtype=np.float32),
                    np.zeros(len(data_chunk), dtype=np.float32))
        fcols    = [c for c in self.model.selected_features
                    if c not in ("volatility","OFI") and c in df.columns]
        X        = df[fcols].values.astype(np.float32)
        vol_inner = self.model._predict_vol(X).astype(np.float32)
        ofi_inner = self.model._predict_ofi(X).astype(np.float32)
        N         = len(data_chunk)
        vol_preds = np.zeros(N, dtype=np.float32)
        ofi_preds = np.zeros(N, dtype=np.float32)
        offset    = N - len(vol_inner)          # warmup rows backfilled
        vol_preds[offset:] = vol_inner
        ofi_preds[offset:] = ofi_inner
        if offset > 0 and len(vol_inner) > 0:
            vol_preds[:offset] = vol_inner[0]
            ofi_preds[:offset] = ofi_inner[0]
        return vol_preds, ofi_preds

    def run(self, data_files, asset, gamma_mapper, mm_strategy, denorm_params, total_rows):
        engine          = ChunkedBacktestEngine(mm_strategy, self.maker_fee,
                                                 self.initial_cash, self.max_inventory)
        rows_per_second = total_rows / (90.0 * 86400.0)
        print(f'   [{asset}] rows_per_second={rows_per_second:.2f}')
        inference_times = []
        global_idx      = 0
        all_pnl_dfs     = []
        all_fills       = []

        for filepath in data_files:
            file_data = torch.load(filepath, map_location="cpu")
            if isinstance(file_data, torch.Tensor): file_data = file_data.numpy()
            print(f"   [{asset}] {os.path.basename(filepath)}  rows={len(file_data):,}")

            for cs in range(0, len(file_data), ROWS_PER_CHUNK):
                chunk = file_data[cs:cs+ROWS_PER_CHUNK].astype(np.float32)
                t0 = time.perf_counter()
                vol_p, ofi_p = self._batch_tree_inference(chunk)
                t_inf = time.perf_counter() - t0
                inference_times.append(t_inf)
                latency_rows = max(1, int(t_inf * rows_per_second))
                arrays, fills = vectorized_backtest_chunk(
                    chunk, vol_p, ofi_p, engine, gamma_mapper, mm_strategy,
                    global_idx, total_rows, denorm_params=denorm_params, latency_rows=latency_rows)
                pnl_arr, pos_arr, gamma_arr = arrays
                chunk_df = pd.DataFrame({
                    "t": (global_idx + np.arange(len(pnl_arr))).astype(np.int32),
                    "pnl": pnl_arr.astype(np.float32),
                    "inventory": pos_arr.astype(np.float32),
                    "gamma": gamma_arr.astype(np.float32),
                })
                all_pnl_dfs.append(chunk_df)
                all_fills.extend(fills)
                global_idx += len(chunk)
                del chunk, chunk_df, arrays, fills

            del file_data; gc.collect()
            try:
                                if torch.cuda.is_available(): torch.cuda.empty_cache()
            except Exception: pass
            try:
                                cp.get_default_memory_pool().free_all_blocks()
            except Exception: pass

        pnl_df   = pd.concat(all_pnl_dfs, ignore_index=True)
        fills_df = pd.DataFrame(all_fills)
        return {"pnl_df":pnl_df, "fills_df":fills_df,
                "inference_times_s":np.array(inference_times),
                "engine_metrics":engine.get_summary_metrics(),
                "model_type":"tree", "asset":asset, "total_rows":total_rows}

print("✅ TreeBacktestRunner defined")


✅ TreeBacktestRunner defined


## Cell 9 — Comprehensive Metrics + Statistical Tests


In [9]:
import numpy as np, pandas as pd
from scipy import stats as scipy_stats
import itertools

def compute_all_metrics(run_result, initial_cash=100_000, trading_days_per_year=252):
    """
    Compute ALL metrics from Part A of the spec.
    """
    pnl_df   = run_result["pnl_df"]
    fills_df = run_result["fills_df"]
    inf_s    = run_result["inference_times_s"]   # seconds per chunk
    em       = run_result["engine_metrics"]

    pnl_series   = pnl_df["pnl"].values        # PnL vs t=0
    inv_series   = pnl_df["inventory"].values
    gamma_series = pnl_df["gamma"].values if "gamma" in pnl_df.columns else np.array([])

    N_rows        = len(pnl_series)
    
    # ---> THE FIX: Use true total rows for fill rate calculation <---
    true_total_rows = run_result.get('total_rows', N_rows)
    
    rows_per_day  = max(1, N_rows // trading_days_per_year)
    daily_pnl     = pnl_series[::rows_per_day]   # approximate daily samples
    daily_returns = np.diff(daily_pnl) / initial_cash   # fractional returns
    n_days        = len(daily_returns)

    # ── Profitability ─────────────────────────────────────────────────────────
    final_pnl        = float(pnl_series[-1])
    total_return_pct = float(final_pnl / initial_cash * 100)
    # Annualize: scale to 252 days regardless of actual period
    actual_days      = max(1, n_days)
    ann_factor       = trading_days_per_year / actual_days
    annualized_return = float(total_return_pct * ann_factor)
    final_cash       = float(em["final_cash"])

    # ── Risk / Drawdown ───────────────────────────────────────────────────────
    daily_vol_pct    = float(np.std(daily_returns) * 100) if n_days > 1 else 0.0
    sharpe_ratio     = float(np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)) if np.std(daily_returns) > 0 else 0.0
    
    downside_returns = daily_returns[daily_returns < 0]
    downside_std     = float(np.std(downside_returns)) if len(downside_returns) > 0 else 0.0
    sortino_ratio    = float(np.mean(daily_returns) / downside_std * np.sqrt(252)) if downside_std > 0 else float('inf')
    
    win_rate         = float(np.mean(daily_returns > 0) * 100) if n_days > 0 else 0.0

    mtm_series        = pnl_series + initial_cash   # absolute portfolio value
    running_max       = np.maximum.accumulate(mtm_series)
    drawdown_abs      = mtm_series - running_max
    drawdown_pct_arr  = drawdown_abs / running_max * 100
    max_dd_dollars    = float(drawdown_abs.min())
    max_dd_pct        = float(drawdown_pct_arr.min())
    calmar_ratio      = float(annualized_return / abs(max_dd_pct)) if abs(max_dd_pct)>0 else 0.0

    # Drawdown duration — rows in deepest drawdown event
    in_dd = drawdown_abs < 0
    if in_dd.any():
        changes       = np.diff(in_dd.astype(int), prepend=0, append=0)
        starts        = np.where(changes == 1)[0]
        ends          = np.where(changes == -1)[0]
        dd_lengths    = ends - starts
        max_dd_rows   = int(dd_lengths.max()) if len(dd_lengths)>0 else 0
        dd_duration_days = float(max_dd_rows / rows_per_day)
    else:
        dd_duration_days = 0.0

    # ── Execution quality ─────────────────────────────────────────────────────
    total_fills    = int(em["total_fills"])
    total_fees     = float(em["total_fees_paid"])
    adverse_pct    = float(em["adverse_selection_pct"])
    net_pnl        = float(final_pnl - total_fees)
    bid_fills      = int(em["bid_fills"]); ask_fills = int(em["ask_fills"])

    if total_fills > 0 and len(fills_df)>0:
        avg_trade_size = float(fills_df.get("quantity", pd.Series([1.0]*len(fills_df))).mean())
        
        # ---> THE FIX: Apply true_total_rows here <---
        fill_rate      = float(total_fills / max(1, true_total_rows) * 1000)  # fills per 1000 rows
        
        if "price" in fills_df.columns and "markout" in fills_df.columns:
            avg_fill_vs_mid_bps = float((fills_df["markout"].abs().mean()) * 10_000)
        else:
            avg_fill_vs_mid_bps = 0.0
    else:
        avg_trade_size = 1.0; fill_rate = 0.0; avg_fill_vs_mid_bps = 0.0

    # ── Inventory management ──────────────────────────────────────────────────
    peak_inv         = float(np.abs(inv_series).max())
    avg_inv          = float(np.abs(inv_series).mean())
    final_inv        = float(em["final_inventory"])
    inventory_turnover = float(total_fills / avg_inv) if avg_inv>0 else 0.0

    # Estimate liquidation value at end
    denorm_params = run_result.get("denorm_params", None)
    if denorm_params and len(pnl_df)>0:
        last_norm_mid = pnl_df.get("norm_mid", pd.Series([0.0])).iloc[-1]
        liquidation_val = float(final_inv * np.exp(last_norm_mid * denorm_params["log_price_std"]
                                                    + denorm_params["log_price_mean"]))
    else:
        liquidation_val = 0.0   # can only be computed if mid price is stored

    # ── Market-making specific ────────────────────────────────────────────────
    if len(gamma_series) > 0:
        gamma_min = float(gamma_series.min()); gamma_max = float(gamma_series.max())
        avg_gamma = float(gamma_series.mean())
    else:
        gamma_min = gamma_max = avg_gamma = float("nan")

    # ── Inference latency ─────────────────────────────────────────────────────
    if len(inf_s) > 0:
        inf_ms = inf_s * 1000  # convert to ms
        inf_mean_ms = float(np.mean(inf_ms))
        inf_p95_ms  = float(np.percentile(inf_ms, 95))
        inf_p99_ms  = float(np.percentile(inf_ms, 99))
        inf_max_ms  = float(np.max(inf_ms))
    else:
        inf_mean_ms = inf_p95_ms = inf_p99_ms = inf_max_ms = float("nan")

    return {
        "final_pnl":              final_pnl,
        "total_return_pct":       total_return_pct,
        "annualized_return_pct":  annualized_return,
        "initial_cash":           initial_cash,
        "final_cash":             final_cash,
        "sharpe_ratio":           sharpe_ratio,
        "sortino_ratio":          sortino_ratio,
        "calmar_ratio":           calmar_ratio,
        "max_drawdown_dollars":   max_dd_dollars,
        "max_drawdown_pct":       max_dd_pct,
        "drawdown_duration_days": dd_duration_days,
        "daily_volatility_pct":   daily_vol_pct,
        "win_rate_pct":           win_rate,
        "total_fills":            total_fills,
        "bid_fills":              bid_fills,
        "ask_fills":              ask_fills,
        "fill_rate_per_1k_rows":  fill_rate,
        "avg_fill_vs_mid_bps":    avg_fill_vs_mid_bps,
        "adverse_selection_pct":  adverse_pct,
        "total_fees_paid":        total_fees,
        "net_pnl_after_fees":     net_pnl,
        "avg_trade_size":         avg_trade_size,
        "peak_inventory":         peak_inv,
        "avg_inventory":          avg_inv,
        "inventory_turnover":     inventory_turnover,
        "final_inventory":        final_inv,
        "liquidation_value":      liquidation_val,
        "gamma_min":              gamma_min,
        "gamma_max":              gamma_max,
        "avg_gamma_t":            avg_gamma,
        "inference_latency_mean_ms": inf_mean_ms,
        "inference_latency_p95_ms":  inf_p95_ms,
        "inference_latency_p99_ms":  inf_p99_ms,
        "max_inference_latency_ms":  inf_max_ms,
        "omega_ratio":         float(np.sum(daily_returns > 0) / np.sum(daily_returns < 0)) if np.sum(daily_returns < 0) > 0 else float("inf"),
        "pct_profitable_days": float(np.mean(daily_returns > 0) * 100) if len(daily_returns) > 0 else 0.0,
        "avg_win_pct":         float(np.mean(daily_returns[daily_returns > 0]) * 100) if np.any(daily_returns > 0) else 0.0,
        "avg_loss_pct":        float(np.abs(np.mean(daily_returns[daily_returns < 0])) * 100) if np.any(daily_returns < 0) else 0.0,
        "win_loss_ratio":      float(np.mean(daily_returns[daily_returns > 0]) / (np.abs(np.mean(daily_returns[daily_returns < 0])) + 1e-8)) if (np.any(daily_returns > 0) and np.any(daily_returns < 0)) else 0.0,
        "consecutive_wins":    int(max([len(list(g)) for k, g in itertools.groupby((daily_returns > 0).astype(int)) if k == 1] or [0])),
        "consecutive_losses":  int(max([len(list(g)) for k, g in itertools.groupby((daily_returns > 0).astype(int)) if k == 0] or [0])),
    }


def bootstrap_sharpe_ci(daily_returns, n_bootstrap=2000, ci=0.95):
    """Bootstrap confidence interval on annualised Sharpe ratio."""
    sharpes, n = [], len(daily_returns)
    for _ in range(n_bootstrap):
        s = np.random.choice(daily_returns, size=n, replace=True)
        if s.std() > 0:
            sharpes.append(s.mean() / s.std() * np.sqrt(252))
    sharpes = np.sort(sharpes)
    alpha = 1 - ci
    lo = np.percentile(sharpes, alpha/2*100)
    hi = np.percentile(sharpes, (1-alpha/2)*100)
    return float(lo), float(hi)


def sharpe_pvalue(ret_a, ret_b, n_bootstrap=5000):
    """
    Permutation p-value for H0: Sharpe(A) == Sharpe(B).
    Returns p-value (two-sided).
    """
    def _sharpe(r): return r.mean()/r.std()*np.sqrt(252) if r.std()>0 else 0.0
    observed = abs(_sharpe(ret_a) - _sharpe(ret_b))
    combined = np.concatenate([ret_a, ret_b]); n_a = len(ret_a)
    diffs = []
    for _ in range(n_bootstrap):
        perm = np.random.permutation(combined)
        diffs.append(abs(_sharpe(perm[:n_a]) - _sharpe(perm[n_a:])))
    return float(np.mean(np.array(diffs) >= observed))

print("✅ compute_all_metrics() updated for OOM safety")


✅ compute_all_metrics() updated for OOM safety


## Cell 10 — Load All Models (3 LSTM + 3 Tree)
> All six models loaded upfront.


In [10]:
import torch

native_tree_models = {}
for asset in ASSETS:
    tp = TREE_MODEL_PATHS[asset]
    print(f"Loading {asset} Tree from {tp['vol']}...")
    m = TreeMarketModel.from_saved(tp['vol'], tp['ofi'], tp['features'])
    native_tree_models[asset] = m
    print(f"  {asset} Tree: {len(m.selected_features)} features")

print(f"All models loaded: Tree={list(native_tree_models.keys())}")


Loading BTC Tree from /workspace/data/14-8-26(btc)/tree/vol_model_final.txt...
✅ TreeMarketModel loaded  — 53 features
   Vol: /workspace/data/14-8-26(btc)/tree/vol_model_final.txt
   OFI: /workspace/data/14-8-26(btc)/tree/ofi_model_final.txt
  BTC Tree: 53 features
Loading ETH Tree from /workspace/data/15-8-26(eth)/tree/vol_model_final.txt...
✅ TreeMarketModel loaded  — 50 features
   Vol: /workspace/data/15-8-26(eth)/tree/vol_model_final.txt
   OFI: /workspace/data/15-8-26(eth)/tree/ofi_model_final.txt
  ETH Tree: 50 features
Loading SOL Tree from /workspace/data/15-8-26(sol)/tree/vol_model_final.txt...
✅ TreeMarketModel loaded  — 58 features
   Vol: /workspace/data/15-8-26(sol)/tree/vol_model_final.txt
   OFI: /workspace/data/15-8-26(sol)/tree/ofi_model_final.txt
  SOL Tree: 58 features
All models loaded: Tree=['BTC', 'ETH', 'SOL']


## Cell 11 — Phase 1: Native Backtests (6 runs)
> BTC/ETH/SOL × LSTM/Tree, each on its own asset.


In [11]:
import glob, os, torch

# =============================================================================
# PHASE 1: NATIVE BACKTESTS — BTC/ETH/SOL x LSTM/Tree = 3 experiments
# Each asset tested with its own trained model.
# =============================================================================

all_results = {}
all_metrics = {}

def _get_asset_files(asset):
    # Use per-asset DATA_PATHS dict
    data_dir = DATA_PATHS.get(asset.upper(),
                              DATA_PATHS.get(list(DATA_PATHS.keys())[0]))
    pattern  = ASSET_GLOB.get(asset.upper(), 'tensor_OFI_Enhanced')
    all_pt   = sorted(glob.glob(os.path.join(data_dir, '*.pt')))
    files    = [f for f in all_pt if pattern in os.path.basename(f)]
    if not files:   # fallback: use ALL .pt files in the folder
        files = all_pt
    if YEAR_FILTER:
        files = [f for f in files if YEAR_FILTER in os.path.basename(f)]
    if not files:
        print('  WARNING: no files found for ' + asset + ' in ' + data_dir)
        print('  Files in dir: ' + str([os.path.basename(x) for x in all_pt[:3]]))
    return files

def _count_rows(files):
    total = 0
    for fp in files:
        try:
            t = torch.load(fp, map_location='cpu', weights_only=True)
        except Exception:
            t = torch.load(fp, map_location='cpu')  # fallback for older .pt files
        total += (t.shape[0] if isinstance(t, torch.Tensor) else len(t))
        del t
    return total

def _run_one(asset, model, model_type, data_files, total_rows):
    cfg          = ASSET_CONFIG.get(asset.upper(), ASSET_CONFIG['BTC'])
    gamma_mapper = GammaMapper(
        gamma_base=cfg['gamma_base'], vol_baseline=cfg['vol_baseline'],
        ofi_sensitivity=5.0, vol_sensitivity=1.0, gamma_min=0.01, gamma_max=1.0)
    mm_strategy = AvellanedaStoikovMarketMaker(
        gamma_mapper=gamma_mapper, k=15.0, T=3600,
        tick_size=cfg['tick_size'], max_inventory=MAX_INVENTORY)
    denorm_params = get_norm_params(asset)
    RunnerCls     = LSTMBacktestRunner if model_type == 'lstm' else TreeBacktestRunner
    runner        = RunnerCls(model=model, initial_cash=INITIAL_CASH,
                              maker_fee=MAKER_FEE, max_inventory=MAX_INVENTORY)
    run_result    = runner.run(data_files=data_files, asset=asset,
                               gamma_mapper=gamma_mapper, mm_strategy=mm_strategy,
                               denorm_params=denorm_params, total_rows=total_rows)
    run_result['denorm_params'] = denorm_params
    run_result['model_type']    = model_type
    run_result['asset']         = asset
    metrics = compute_all_metrics(run_result, initial_cash=INITIAL_CASH)
    metrics['asset']      = asset
    metrics['model_type'] = model_type
    return run_result, metrics

for asset in ASSETS:
    files      = _get_asset_files(asset)
    total_rows = _count_rows(files)
    if not files:
        print(f"No files for {asset} - skipping")
        continue
    print(f"\n{'='*70}")
    print(f"  PHASE 1 | {asset} | Files={len(files)} | Rows={total_rows:,}")
    print(f"{'='*70}")
    for model_type, native_models in [('tree', native_tree_models)]:
        model = native_models[asset]
        print(f"\n  [{asset}] {model_type.upper()} model...")
        run_result, metrics = _run_one(asset, model, model_type, files, total_rows)
        key = f"{asset}_{model_type}"
        all_results[key] = run_result
        all_metrics[key] = metrics
        print(f"   Sharpe={metrics['sharpe_ratio']:.3f}  Sortino={metrics['sortino_ratio']:.3f}  "
              f"Calmar={metrics['calmar_ratio']:.3f}  Omega={metrics['omega_ratio']:.3f}")
        print(f"   MaxDD={metrics['max_drawdown_pct']:.2f}%  Fills={metrics['total_fills']:,}  "
              f"AdvSel={metrics['adverse_selection_pct']:.1f}%  ExecLat=100ms")

print(f"\nPhase 1 done — {len(all_results)} experiments completed.")
print("Next: Cell 12 (save) -> Cell 13 (compare) -> Cell 14 (micro) -> Cell 15 (transfer) -> Cell 16 (figures)")



  PHASE 1 | BTC | Files=90 | Rows=142,383,659

  [BTC] TREE model...
   Using fallback norm params for BTC: {'log_price_mean': 10.861594, 'log_price_std': 0.010742}
   [BTC] rows_per_second=18.31
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-01.pt  rows=761,222
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-02.pt  rows=1,677,641
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-03.pt  rows=2,295,158
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-04.pt  rows=1,411,787
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-05.pt  rows=1,617,499
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-06.pt  rows=656,685
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-07.pt  rows=785,460
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-08.pt  rows=1,943,603
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-09.pt  rows=2,227,000
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-10.pt  rows=3,014,236
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-11.pt  rows=2,764,196
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-12.pt  rows=2,403,360
   [BTC] tensor_OFI_Enhanced_BTC_2024-01-13.pt  rows=

## Cell 12 — Save Results to Disk


In [12]:
import json, os, gc

RESULTS_DIR = "backtest_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Writing massive Phase 1 data to disk in memory-safe chunks...")

for key, run_result in all_results.items():
    print(f"  Saving {key}...")
    
    # Save PnL using chunksize to prevent string-conversion memory spikes
    pnl_path = os.path.join(RESULTS_DIR, f"{key}_pnl.csv")
    if "pnl_df" in run_result:
        run_result["pnl_df"].to_csv(pnl_path, index=False, chunksize=200000)
        
        # FREE UP RAM IMMEDIATELY! (This drops your 13.5 GB back down)
        del run_result["pnl_df"]
        gc.collect()
    
    # Save fills using chunksize
    fills_path = os.path.join(RESULTS_DIR, f"{key}_fills.csv")
    if "fills_df" in run_result:
        run_result["fills_df"].to_csv(fills_path, index=False, chunksize=200000)
        
        # FREE UP RAM IMMEDIATELY!
        del run_result["fills_df"]
        gc.collect()

# Save metrics as JSON
metrics_path = os.path.join(RESULTS_DIR, "metrics_tree.json")
serializable = {k: {mk: (float(mv) if not isinstance(mv, str) else mv)
                     for mk, mv in mdict.items()}
                for k, mdict in all_metrics.items()}
with open(metrics_path, "w") as f:
    json.dump(serializable, f, indent=2)

print(f"✅ Finished writing all CSVs. Raw data successfully flushed from RAM!")
print(f"✅ You are now perfectly safe to begin Phase 2 with a clean slate of memory!")


Writing massive Phase 1 data to disk in memory-safe chunks...
  Saving BTC_tree...
  Saving ETH_tree...
  Saving SOL_tree...
✅ Finished writing all CSVs. Raw data successfully flushed from RAM!
✅ You are now perfectly safe to begin Phase 2 with a clean slate of memory!


In [13]:
import json, os, numpy as np, pandas as pd

RESULTS_DIR = "backtest_results"

# ── Load both models' metrics ─────────────────────────────────────────────────
def _load_metrics(model_type):
    path = os.path.join(RESULTS_DIR, f"metrics_{model_type}.json")
    if not os.path.exists(path):
        print(f"⚠️  {path} not found — run Cell 11+12 with USE_MODEL='{model_type}' first.")
        return {}
    with open(path) as f:
        return json.load(f)

metrics_lstm = _load_metrics("lstm")
metrics_tree = _load_metrics("tree")

# ── Bootstrap CI (reload daily returns from saved PnL) ───────────────────────
def _daily_returns(pnl_csv, initial_cash, rows_per_day=None):
    df = pd.read_csv(pnl_csv)
    pnl = df["pnl"].values
    if rows_per_day is None:
        rows_per_day = max(1, len(pnl) // 252)
    daily = pnl[::rows_per_day]
    return np.diff(daily) / initial_cash

print("=" * 90)
print(f"  RAMM BACKTEST — LSTM vs TREE COMPARISON")
print("=" * 90)

METRICS_TO_COMPARE = [
    ("final_pnl",                 "Final PnL ($)",            "${:>12,.2f}",    True),
    ("total_return_pct",          "Return (%)",               "{:>10.2f}%",     True),
    ("annualized_return_pct",     "Ann. Return (%)",          "{:>10.2f}%",     True),
    ("sharpe_ratio",              "Sharpe Ratio",             "{:>12.3f}",      True),
    ("sortino_ratio",             "Sortino Ratio",            "{:>12.3f}",      True),
    ("calmar_ratio",              "Calmar Ratio",             "{:>12.3f}",      True),
    ("max_drawdown_pct",          "Max Drawdown (%)",         "{:>10.2f}%",     False),
    ("max_drawdown_dollars",      "Max Drawdown ($)",         "${:>12,.2f}",    False),
    ("drawdown_duration_days",    "DD Duration (days)",       "{:>10.1f}",      False),
    ("daily_volatility_pct",      "Daily Vol (%)",            "{:>10.3f}%",     False),
    ("win_rate_pct",              "Win Rate (%)",             "{:>10.1f}%",     True),
    ("total_fills",               "Total Fills",              "{:>12,.0f}",     True),
    ("fill_rate_per_1k_rows",     "Fill Rate (per 1k rows)",  "{:>10.4f}",      True),
    ("adverse_selection_pct",     "Adverse Sel (%)",          "{:>10.1f}%",     False),
    ("total_fees_paid",           "Total Fees ($)",           "${:>12,.2f}",    False),
    ("net_pnl_after_fees",        "Net PnL after Fees ($)",   "${:>12,.2f}",    True),
    ("peak_inventory",            "Peak Inventory",           "{:>12.2f}",      False),
    ("avg_inventory",             "Avg Inventory",            "{:>12.2f}",      False),
    ("avg_gamma_t",               "Avg γ_t",                  "{:>12.4f}",      False),
    ("gamma_min",                 "γ_t Min",                  "{:>12.4f}",      False),
    ("gamma_max",                 "γ_t Max",                  "{:>12.4f}",      False),
    ("inference_latency_mean_ms", "Inf Latency Mean (ms)",    "{:>10.3f}",      False),
    ("inference_latency_p95_ms",  "Inf Latency P95 (ms)",     "{:>10.3f}",      False),
    ("inference_latency_p99_ms",  "Inf Latency P99 (ms)",     "{:>10.3f}",      False),
    ("max_inference_latency_ms",  "Inf Latency Max (ms)",     "{:>10.3f}",      False),
]

for asset in ASSETS:
    key_l = f"{asset}_lstm"; key_t = f"{asset}_tree"
    ml = metrics_lstm.get(key_l, {}); mt = metrics_tree.get(key_t, {})
    if not ml and not mt:
        continue

    print(f"\n{'─'*90}")
    print(f"  ASSET: {asset}")
    print(f"{'─'*90}")
    print(f"  {'Metric':<35}  {'LSTM':>18}  {'Tree':>18}  {'Diff (T-L)':>14}  Win")
    print(f"  {'-'*35}  {'-'*18}  {'-'*18}  {'-'*14}  ---")

    for mkey, label, fmt, higher_better in METRICS_TO_COMPARE:
        vl = ml.get(mkey, float("nan")); vt = mt.get(mkey, float("nan"))
        try:
            sl = fmt.format(vl); st = fmt.format(vt)
            diff = vt - vl; sd = fmt.format(diff).replace("$","").strip()
            if higher_better:
                win = "✅ Tree" if vt > vl else ("✅ LSTM" if vl > vt else "  tie ")
            else:
                win = "✅ Tree" if vt < vl else ("✅ LSTM" if vl < vt else "  tie ")
        except Exception:
            sl = st = sd = "N/A"; win = "  —  "
        print(f"  {label:<35}  {sl:>18}  {st:>18}  {sd:>14}  {win}")

    # ── Bootstrap CI on Sharpe ────────────────────────────────────────────────
    pnl_l = os.path.join(RESULTS_DIR, f"{key_l}_pnl.csv")
    pnl_t = os.path.join(RESULTS_DIR, f"{key_t}_pnl.csv")
    if os.path.exists(pnl_l) and os.path.exists(pnl_t):
        ret_l = _daily_returns(pnl_l, INITIAL_CASH)
        ret_t = _daily_returns(pnl_t, INITIAL_CASH)
        if len(ret_l)>10 and len(ret_t)>10:
            ci_lo_l, ci_hi_l = bootstrap_sharpe_ci(ret_l)
            ci_lo_t, ci_hi_t = bootstrap_sharpe_ci(ret_t)
            pval = sharpe_pvalue(ret_l, ret_t)
            print(f"\n  📊 Bootstrap Sharpe CI (95%):")
            print(f"     LSTM  Sharpe: {ml.get('sharpe_ratio',float('nan')):.3f}  [{ci_lo_l:.3f}, {ci_hi_l:.3f}]")
            print(f"     Tree  Sharpe: {mt.get('sharpe_ratio',float('nan')):.3f}  [{ci_lo_t:.3f}, {ci_hi_t:.3f}]")
            print(f"     Permutation p-value (H0: no difference): {pval:.4f}", end="")
            if pval < 0.01: print(" *** (p<0.01)")
            elif pval < 0.05: print(" **  (p<0.05)")
            elif pval < 0.10: print(" *   (p<0.10)")
            else: print(" (not significant)")

# ── Save comparison CSV ───────────────────────────────────────────────────────
print(f"\n{'='*90}")
rows_out = []
for asset in ASSETS:
    key_l = f"{asset}_lstm"; key_t = f"{asset}_tree"
    ml = metrics_lstm.get(key_l,{}); mt = metrics_tree.get(key_t,{})
    for mkey, label, _, _ in METRICS_TO_COMPARE:
        rows_out.append({"asset":asset, "metric":label,
                          "lstm":ml.get(mkey,""), "tree":mt.get(mkey,"")})
if rows_out:
    df_out = pd.DataFrame(rows_out)
    df_out.to_csv(os.path.join(RESULTS_DIR, "comparison_lstm_vs_tree.csv"), index=False)
    print(f"✅ Comparison saved → {RESULTS_DIR}/comparison_lstm_vs_tree.csv")


⚠️  backtest_results/metrics_lstm.json not found — run Cell 11+12 with USE_MODEL='lstm' first.
  RAMM BACKTEST — LSTM vs TREE COMPARISON

──────────────────────────────────────────────────────────────────────────────────────────
  ASSET: BTC
──────────────────────────────────────────────────────────────────────────────────────────
  Metric                                             LSTM                Tree      Diff (T-L)  Win
  -----------------------------------  ------------------  ------------------  --------------  ---
  Final PnL ($)                             $         nan       $   61,140.48             nan    tie 
  Return (%)                                         nan%               0.61%            nan%    tie 
  Ann. Return (%)                                    nan%               0.61%            nan%    tie 
  Sharpe Ratio                                        nan               1.293             nan    tie 
  Sortino Ratio                                       nan    

## Cell 14 — Microstructure Analysis
> Spread, OFI distribution, volume stats per asset.


In [14]:
import numpy as np, pandas as pd, torch, os
from scipy import stats as scipy_stats

def compute_microstructure(asset):
    files = _get_asset_files(asset)
    if not files: return None
    data = torch.load(files[0], map_location='cpu', weights_only=True)
    if isinstance(data, torch.Tensor): data = data.numpy()
    spread = data[:,2]; ofi = data[:,3]; vol = data[:,5]; volume = data[:,1]; pc = data[:,6]
    return {
        'asset':            asset,
        'n_rows':           len(data),
        'spread_mean':      float(np.mean(spread)),
        'spread_median':    float(np.median(spread)),
        'spread_std':       float(np.std(spread)),
        'spread_max':       float(np.max(spread)),
        'vol_mean':         float(np.mean(vol)),
        'vol_std':          float(np.std(vol)),
        'ofi_mean':         float(np.mean(ofi)),
        'ofi_std':          float(np.std(ofi)),
        'ofi_skewness':     float(scipy_stats.skew(ofi)),
        'ofi_kurtosis':     float(scipy_stats.kurtosis(ofi)),
        'volume_mean':      float(np.mean(volume)),
        'volume_std':       float(np.std(volume)),
        'price_change_std': float(np.std(pc)),
    }

micro_rows = [compute_microstructure(a) for a in ASSETS]
micro_rows = [r for r in micro_rows if r]
micro_df   = pd.DataFrame(micro_rows)

print("="*80)
print("MICROSTRUCTURE ANALYSIS (first file per asset)")
print("="*80)
print(micro_df.set_index('asset').T.to_string())

os.makedirs('backtest_results', exist_ok=True)

# Distribution shift analysis (train vs test)
print("\n" + "="*80)
print("DISTRIBUTION SHIFT: TRAIN vs TEST (log-price)")
print("="*80)
print("  {:<6}  {:>16}  {:>16}  {:>14}  {:>14}".format(
    "Asset", "Train Mean", "Test Mean", "Mean Shift", "Std Ratio"))
print("  " + "-"*70)
for asset in ASSETS:
    tr = NORM_PARAMS_TRAIN.get(asset, {})
    te = NORM_PARAMS_TEST.get(asset, {})
    if tr and te:
        mean_shift = te["log_price_mean"] - tr["log_price_mean"]
        std_ratio  = te["log_price_std"]  / tr["log_price_std"]
        mid_train  = __import__("math").exp(tr["log_price_mean"])
        mid_test   = __import__("math").exp(te["log_price_mean"])
        print("  {:<6}  {:>16.6f}  {:>16.6f}  {:>14.6f}  {:>14.4f}".format(
            asset,
            tr["log_price_mean"], te["log_price_mean"],
            mean_shift, std_ratio))
        print("         (mid ~${:,.0f})     (mid ~${:,.0f})".format(mid_train, mid_test))
print()
print("  Positive mean shift = asset price HIGHER in test period than training period")
print("  Std ratio > 1 = HIGHER volatility in test period (harder for model)")
print("  Std ratio < 1 = LOWER volatility in test period (easier for model)")
micro_df.to_csv('backtest_results/microstructure_metrics.csv', index=False)
print("\nSaved -> backtest_results/microstructure_metrics.csv")


MICROSTRUCTURE ANALYSIS (first file per asset)
asset                       BTC            ETH           SOL
n_rows            761222.000000  504953.000000  1.011143e+06
spread_mean            0.101487       0.116248  9.970543e-01
spread_median          0.023525       0.043448  2.920108e-01
spread_std             0.315057       0.379480  1.728541e+00
spread_max             6.524006      14.744660  3.575770e+01
vol_mean               0.000102       0.000101  1.133693e-04
vol_std                0.000024       0.000013  5.959889e-05
ofi_mean               0.100916       0.057164 -1.021176e-03
ofi_std                0.999999       0.999999  9.999995e-01
ofi_skewness           0.586251       0.710107  3.751900e-01
ofi_kurtosis           2.664699       3.336243  3.002781e+00
volume_mean            1.000000       1.000000  1.000000e+00
volume_std             4.547489       4.180496  3.720891e+00
price_change_std       0.000044       0.000038  1.086304e-04

DISTRIBUTION SHIFT: TRAIN vs TEST (lo

## Cell 15 — Phase 2: Cross-Asset Transfer Testing (12 runs)
> All 6 source→target pairs × LSTM + Tree. Frozen source weights.


In [15]:
import pandas as pd, numpy as np

# =============================================================================
# PHASE 2: FULL CROSS-ASSET TRANSFER TESTING
# All 6 pairs x 1 model = 6 experiments
# BTC->ETH, BTC->SOL, ETH->BTC, ETH->SOL, SOL->BTC, SOL->ETH
# Source model FROZEN — tested on target asset data
# =============================================================================

TRANSFER_PAIRS    = [(s,t) for s in ASSETS for t in ASSETS if s != t]
transfer_results  = {}
transfer_metrics  = {}
degradation_table = {}   # (src, tgt, model_type) -> pct

print("="*70)
print("PHASE 2 - CROSS-ASSET TRANSFER TESTING")
print(f"{len(TRANSFER_PAIRS)} pairs x 1 model = {len(TRANSFER_PAIRS)*2} experiments")
print("="*70)

for (src, tgt) in TRANSFER_PAIRS:
    tgt_files = _get_asset_files(tgt)
    tgt_rows  = _count_rows(tgt_files)
    if not tgt_files:
        print(f"No data for {tgt} - skipping"); continue
    print(f"\n{'='*60}")
    print(f"  {src} -> {tgt}  ({tgt_rows:,} rows)")
    for model_type, native_models in [('tree', native_tree_models)]:
        model = native_models[src]   # SOURCE model, FROZEN
        key   = f"{src}_to_{tgt}_{model_type}"
        print(f"  [{src}->{tgt}] {model_type.upper()} (frozen {src} weights)...", end=" ")
        run_result, metrics = _run_one(tgt, model, model_type, tgt_files, tgt_rows)
        metrics['source_asset'] = src
        metrics['target_asset'] = tgt
        metrics['transfer_key'] = key
        native_sharpe   = all_metrics.get(f"{tgt}_{model_type}", {}).get('sharpe_ratio', float('nan'))
        transfer_sharpe = metrics['sharpe_ratio']
        if not (np.isnan(native_sharpe) or native_sharpe == 0):
            degradation_pct = (native_sharpe - transfer_sharpe) / abs(native_sharpe) * 100
        else:
            degradation_pct = float('nan')
        metrics['sharpe_degradation_pct'] = degradation_pct
        metrics['native_sharpe']          = native_sharpe
        transfer_results[key] = run_result
        # IMMEDIATELY save and dump to prevent OOM
        import gc, os
        RESULTS_DIR = "backtest_results"
        os.makedirs(RESULTS_DIR, exist_ok=True)
        if "pnl_df" in run_result and run_result["pnl_df"] is not None:
            pnl_path = os.path.join(RESULTS_DIR, f"{key}_pnl.csv")
            run_result["pnl_df"].to_csv(pnl_path, index=False, chunksize=200000)
            del run_result["pnl_df"]
        if "fills_df" in run_result and run_result["fills_df"] is not None:
            fills_path = os.path.join(RESULTS_DIR, f"{key}_fills.csv")
            run_result["fills_df"].to_csv(fills_path, index=False, chunksize=200000)
            del run_result["fills_df"]
        gc.collect()
        transfer_metrics[key] = metrics
        degradation_table[(src, tgt, model_type)] = degradation_pct
        deg = f"{degradation_pct:.1f}%" if not np.isnan(degradation_pct) else "N/A"
        print(f"Sharpe={transfer_sharpe:.3f} (native={native_sharpe:.3f}, degradation={deg})")

# Print 3x3 degradation matrices
print("\n" + "="*70)
print("TRANSFER DEGRADATION MATRIX (Sharpe drop % — positive = worse)")
for mtype in ['lstm', 'tree']:
    mat = pd.DataFrame(index=ASSETS, columns=ASSETS, dtype=float)
    for s in ASSETS:
        for t in ASSETS:
            mat.loc[s,t] = 0.0 if s==t else degradation_table.get((s,t,mtype), float('nan'))
    mat.index.name = f"{mtype.upper()} Source/Target"
    print(f"\n{mtype.upper()}:"); print(mat.round(1).to_string())

pd.DataFrame(list(transfer_metrics.values())).to_csv(
    'backtest_results/thesis_transfer_results.csv', index=False)
print("\nSaved -> backtest_results/thesis_transfer_results.csv")


PHASE 2 - CROSS-ASSET TRANSFER TESTING
6 pairs x 1 model = 12 experiments

  BTC -> ETH  (112,311,203 rows)
  [BTC->ETH] TREE (frozen BTC weights)...    Using fallback norm params for ETH: {'log_price_mean': 7.954236, 'log_price_std': 0.011872}
   [ETH] rows_per_second=14.44
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-01.pt  rows=504,953
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-02.pt  rows=1,056,586
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-03.pt  rows=1,672,933
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-04.pt  rows=846,744
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-05.pt  rows=918,614
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-06.pt  rows=463,908
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-07.pt  rows=533,736
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-08.pt  rows=1,130,205
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-09.pt  rows=1,453,172
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-10.pt  rows=2,577,691
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-11.pt  rows=1,879,136
   [ETH] tensor_OFI_Enhanced_ETH_2024-01-

## Cell 16 — Figures & Thesis Summary
> 6 diagrams + Phase 1 & Phase 2 CSV tables.


In [ ]:
import pandas as pd, numpy as np, gc, os

# =============================================================================
# PHASE 2: CROSS-ASSET TRANSFER TESTING (100% OOM SAFE)
# =============================================================================

RESULTS_DIR = "backtest_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. PURGE PHASE 1 FROM RAM BEFORE STARTING PHASE 2
if 'all_results' in globals():
    print("Purging Phase 1 arrays from RAM...")
    del all_results
    for _ in range(3): gc.collect()

TRANSFER_PAIRS = [(s,t) for s in ASSETS for t in ASSETS if s != t]
transfer_metrics = {}
degradation_table = {}

print("="*70)
print("PHASE 2 - CROSS-ASSET TRANSFER TESTING")
print(f"{len(TRANSFER_PAIRS)} pairs x 1 model = {len(TRANSFER_PAIRS)} experiments")
print("="*70)

for (src, tgt) in TRANSFER_PAIRS:
    tgt_files = _get_asset_files(tgt)
    tgt_rows  = _count_rows(tgt_files)
    if not tgt_files: continue
    
    print(f"\n{'='*60}")
    print(f"  {src} -> {tgt}  ({tgt_rows:,} rows)")
    
    model = native_tree_models[src]   # SOURCE model, FROZEN
    key = f"{src}_to_{tgt}_tree"
    print(f"  [{src}->{tgt}] TREE (frozen {src} weights)...", end=" ")
    
    # Run backtest
    run_result, metrics = _run_one(tgt, model, 'tree', tgt_files, tgt_rows)
    
    # Compile metrics
    metrics['source_asset'] = src
    metrics['target_asset'] = tgt
    metrics['transfer_key'] = key
    
    native_sharpe = all_metrics.get(f"{tgt}_tree", {}).get('sharpe_ratio', float('nan')) if 'all_metrics' in globals() else float('nan')
    transfer_sharpe = metrics['sharpe_ratio']
    
    if not (np.isnan(native_sharpe) or native_sharpe == 0):
        degradation_pct = (native_sharpe - transfer_sharpe) / abs(native_sharpe) * 100
    else:
        degradation_pct = float('nan')
        
    metrics['sharpe_degradation_pct'] = degradation_pct
    metrics['native_sharpe'] = native_sharpe
    
    transfer_metrics[key] = metrics
    degradation_table[(src, tgt, 'tree')] = degradation_pct
    
    # 2. SAVE AND PURGE PNL/FILLS TO DISK
    if "pnl_df" in run_result and run_result["pnl_df"] is not None:
        run_result["pnl_df"].to_csv(os.path.join(RESULTS_DIR, f"{key}_pnl.csv"), index=False, chunksize=200000)
        del run_result["pnl_df"] 
        
    if "fills_df" in run_result and run_result["fills_df"] is not None:
        run_result["fills_df"].to_csv(os.path.join(RESULTS_DIR, f"{key}_fills.csv"), index=False, chunksize=200000)
        del run_result["fills_df"] 
        
    del run_result
    for _ in range(3): gc.collect()
    
    deg = f"{degradation_pct:.1f}%" if not np.isnan(degradation_pct) else "N/A"
    print(f"Sharpe={transfer_sharpe:.3f} (native={native_sharpe:.3f}, degradation={deg})")

# =============================================================================
# SUMMARY TABLES (Moved here so you get your final outputs safely)
# =============================================================================
print("\n" + "="*80)
print("TRANSFER DEGRADATION MATRIX (Sharpe drop % — positive = worse)")
mat = pd.DataFrame(index=ASSETS, columns=ASSETS, dtype=float)
for s in ASSETS:
    for t in ASSETS:
        mat.loc[s,t] = 0.0 if s==t else degradation_table.get((s,t,'tree'), float('nan'))
mat.index.name = "TREE Source/Target"
print(mat.round(1).to_string())

COLS2=['source_asset','target_asset','model_type','sharpe_ratio','native_sharpe',
       'sharpe_degradation_pct','max_drawdown_pct','total_fills','adverse_selection_pct']
p2=pd.DataFrame([{k:m.get(k,'') for k in COLS2} for m in transfer_metrics.values()])
if not p2.empty:
    p2=p2.sort_values(['source_asset','target_asset'])
    print("\nPHASE 2 TRANSFER RESULTS:"); print(p2.to_string(index=False))
    p2.to_csv('backtest_results/thesis_phase2_transfer.csv', index=False)

# Dump raw metrics for safekeeping
pd.DataFrame(list(transfer_metrics.values())).to_csv('backtest_results/thesis_transfer_metrics_tree.csv', index=False)
print("\nPhase 2 Complete! All data securely dumped to backtest_results/ folder.")
